In [ ]:
%pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 4.0 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 4.1 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] 1/2 [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# MobileNetV3 Large model load

import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import json
import time


# MobileNetV3 Large 모델 로드


def load_mobilenet_v3():
    global model, preprocess, EMB_SIZE

    print("MobileNetV3 Large 모델 로드 중...")

    weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
    model = models.mobilenet_v3_large(weights=weights)

    # 최종 FC 제거하여 임베딩 출력
    model.classifier = nn.Identity()
    EMB_SIZE = 960  # MobileNetV3 Large 출력 차원

    preprocess = weights.transforms()  # 자동 Resize/Normalize

    model.eval()

    # ---- warm-up ----
    dummy = torch.zeros(1, 3, 224, 224)
    with torch.no_grad():
        model(dummy)

    print("MobileNetV3 Large Warm-up 완료!")
    print(f"임베딩 차원: {EMB_SIZE}\n")

load_mobilenet_v3()

MobileNetV3 Large 모델 로드 중...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /Users/jeong-ug/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:05<00:00, 3.89MB/s]


MobileNetV3 Large Warm-up 완료!
임베딩 차원: 960



In [3]:
# 이미지 다운로드 + 전처리 함수

def load_and_preprocess(url):
    try:
        response = requests.get(url, timeout=(2, 5))
        response.raise_for_status()

        img = Image.open(BytesIO(response.content)).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0)

        return img_tensor

    except Exception as e:
        print(f"이미지 로드 실패: {url}")
        print("사유:", e)
        return None

In [4]:
# 임베딩 생성

def get_embedding(url):
    img_tensor = load_and_preprocess(url)
    if img_tensor is None:
        return None

    with torch.no_grad():
        emb = model(img_tensor).squeeze().numpy().astype("float32")

    # L2 정규화
    emb = emb / (np.linalg.norm(emb) + 1e-10)

    return emb

In [ ]:
products = [
  {
    "title": "고급형 15.6inch N95 노트북 컴퓨터 국경 간",
    "price": "329,290원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=894979810606&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01vRzqYw1VP0HsuJ0IZ_!!2214200952644-0-cib.jpg",
    "detail_specs": {
      "상표": "아이오파/아이오파",
      "모델": "AP156PC01",
      "품목 번호": "AP156PC01",
      "시장 출시 시간": "2023",
      "공급 카테고리": "스팟상품",
      "브랜드 지역": "국내",
      "CPU 유형": "intel",
      "CPU 주파수": "기계를 보세요",
      "하드 드라이브 용량": "선택 사항",
      "광학 플로피 드라이브 위치": "광학 드라이브 없음",
      "화면 크기": "15.6inch",
      "그래픽 카드": "통합 그래픽",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해결책": "풀 HD(1920x1080)",
      "운영 체제": "win10/win11/Chrome OS",
      "터치스크린 여부": "비터치",
      "두께": "일반 두께(>25mm)",
      "배터리 수명": "5~7시간",
      "제품 크기": "15.6",
      "무게": "1.5-2.0KG",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "8~15일",
      "프록시 가입 지원 여부": "지원하다",
      "판매 후 유형": "3개 보증 저장",
      "보증": "1 년",
      "송장": "송장 제공",
      "포장 목록": "포장 참조",
      "3C 인증서 번호": "2025200902000005",
      "비디오 메모리 용량": "선택 사항",
      "색상": "N95+32GB 실행+2TB 솔리드 스테이트+지문 잠금 해제",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "에너지 효율 수준": "레벨 1",
      "에너지효율등록번호": "20250221-CEL0272016-419705",
      "유형": "얇고 가벼운 휴대용 노트북"
    }
  },
  {
    "title": "16G+512G 태블릿 컴퓨터 패드 프로 풀 네트워크 5G 안드로이드 오피스 엔터테인먼트 게임 학습 기계 투인원",
    "price": "98,260원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=890786635518&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01Kmlu6P1UNLEhQ76qt_!!2219048012505-0-cib.jpg",
    "detail_specs": {
      "모델": "D23",
      "품목 번호": "D23",
      "시장 출시 시간": "2025",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "MTK",
      "프로세서 주파수": "기타 주요 주파수",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "64GB、128GB、256GB、512GB、1TB",
      "화면 크기": "10.1inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원 전화",
      "네트워크 유형": "WIFI",
      "내장 센서": "주변광 감지",
      "프로세서 코어": "8개의 코어",
      "배터리 수명": "5~7시간",
      "추가 기능": "블루투스 지원",
      "언어": "중국어 번체",
      "재료/공정": "금속, 유리, 플라스틱",
      "제품 크기": "246.1*172.1*8.6MM",
      "무게": "400",
      "맞춤형 처리": "예",
      "로고 인쇄": "안됨",
      "프록시 가입 지원 여부": "지원하다",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장 제공",
      "판매 후 유형": "매장 보증",
      "3C 인증서 번호": "2022010902489181",
      "색상": "은",
      "메모리 용량": "16G 운영 + 512G 메모리/고급 버전에는 키보드 가죽 케이스와 스타일러스가 함께 제공됩니다.",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "아니요",
      "통신 유형": "카드 삽입 가능",
      "통신장비망접속허가번호": "17-G990-223329",
      "공급원이 특허화되어 있나요?": "아니요",
      "무선전송장치 모델 승인코드": "1",
      "유형": "비즈니스 태블릿"
    }
  },
  {
    "title": "데스크탑 컴퓨터 i5i7 게이밍 독립 그래픽 카드 풀세트 가정용 사무용 디자인 13세대 14세대 데스크탑 컴퓨터",
    "price": "90,950원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=673687408100&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01knLW8w1eVGB1r9Kne_!!1941143876-0-cib.jpg",
    "detail_specs": {
      "상표": "야만적인 작은 악마",
      "모델": "패키지 1: 호스트 1개 [비즈니스 사무실 및 가정용]",
      "품목 번호": "009",
      "시장 출시 시간": "2023",
      "공급 카테고리": "조립 기계",
      "유형": "게임용 컴퓨터",
      "CPU 유형": "인텔/인텔",
      "CPU 주파수": "3.4GHz",
      "메모리 용량": "8GB",
      "하드 드라이브 용량": "1024GB",
      "모니터 유형": "액정",
      "모니터 크기": "32",
      "그래픽 카드 유형": "별도의 그래픽 카드",
      "OEM": "OEM 불가",
      "보증": "1년",
      "총중량": "1.27",
      "운영 체제": "windows10",
      "가장 빠른 배송 시간": "1~3일",
      "판매 후 유형": "3개 보증 저장",
      "3C 인증서 번호": "2024230901002843",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스가 부여된 개인 상표가 있습니다.": "아니요",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "에너지 효율 수준": "레벨 1"
    }
  },
  {
    "title": "C 아이디어 안드로이드 태블릿 PC 국경",
    "price": "47,990원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=894730026675&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01wkr0xN2HV32NIYLEL_!!2215386249155-0-cib.jpg",
    "detail_specs": {
      "상표": "cidea",
      "모델": "CM516",
      "운영 체제": "ANDROID",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "128GB",
      "화면 크기": "7inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원되지",
      "네트워크 유형": "WIFI",
      "프로세서 코어": "4개의 코어",
      "추가 기능": "IPS 스크린",
      "언어": "다른",
      "재료/공정": "금속",
      "제품 크기": "7inch",
      "맞춤형 처리": "예",
      "로고 인쇄": "할 수 있다",
      "가장 빠른 배송 시간": "4~7일",
      "3C 인증서 번호": "2021011606365783",
      "색상": "검은색",
      "주요 다운스트림 플랫폼": "아마존",
      "주요 판매지역": "중동",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "유형": "학생용 태블릿"
    }
  },
  {
    "title": "15.6inch 노트북 i7 쿼드 코어 독립 그래픽 게임 사무실 비즈니스 학습 온라인 수업 라이브 방송 i5 초박형 휴대용",
    "price": "191,950원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=530803621189&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01A4lQkx2NXgmdHWMct_!!1911179973-0-cib.jpg",
    "detail_specs": {
      "상표": "다른",
      "공급 카테고리": "스팟상품",
      "브랜드 지역": "국내",
      "CPU 유형": "코어/코어 i7",
      "CPU 주파수": "2.4G",
      "하드 드라이브 용량": "250GB",
      "광학 플로피 드라이브 위치": "내장형 광학 드라이브",
      "광학 드라이브 유형": "DVD±RW",
      "화면 크기": "14inch",
      "그래픽 카드": "통합 그래픽",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해결책": "일반 화면(1366×768)",
      "운영 체제": "windows7",
      "터치스크린 여부": "비터치",
      "두께": "보통 및 얇음(21mm-25mm)",
      "배터리 수명": "5시간 이내",
      "제품 크기": "14inch",
      "무게": "2.0-2.5kg",
      "OEM": "OEM 불가",
      "가장 빠른 배송 시간": "1~3일",
      "프록시 가입 지원 여부": "지원하다",
      "판매 후 유형": "3개 보증 저장",
      "보증": "1 년",
      "송장": "송장은 제공되지 않습니다",
      "비디오 메모리 용량": "2G",
      "색상": "구성 23 상위 구성 권장",
      "라이센스가 부여된 개인 상표가 있습니다.": "아니요",
      "국경 간 수출을 위한 독점 공급원인지 여부": "아니요",
      "에너지 효율 수준": "없음",
      "공급원이 특허화되어 있나요?": "아니요",
      "유형": "게임 오디오 및 비디오 책"
    }
  },
  {
    "title": "노트북 840G1G2820G3440G3 비즈니스 노트북 i5i7 휴대용 중고 얇고 가벼운 사무용",
    "price": "113,800원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=800687123720&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN0158k7gQ1jD0PC2r8Yg_!!3855954513-0-cib.jpg",
    "detail_specs": {
      "상표": "H",
      "상품 번호": "840",
      "공급 카테고리": "현물 상품",
      "브랜드 지역": "국내",
      "CPU 유형": "코어/코어 i5",
      "하드 드라이브 용량": "320GB",
      "광학 플로피 드라이브 위치": "광학 드라이브 없음",
      "화면 크기": "14inch",
      "그래픽 카드": "통합 그래픽",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해결책": "일반 화면(1366X768)",
      "운영 체제": "Windows0",
      "터치스크린 여부": "비터치",
      "두께": "일반적으로 얇고 가볍습니다(21mm-25mm).",
      "지속 시간": "5시간 이내",
      "무게": "2.0-2.5kg",
      "OEM": "OEM 없음",
      "가장 빠른 배송 시간": "4~7일",
      "프록시 가맹을 지원할지 여부": "지원하다",
      "판매 후 유형": "3개 보증 저장",
      "보증": "3 개월",
      "송장": "송장은 제공되지 않습니다",
      "포장 목록": "컴퓨터 + 충전기",
      "3C 인증서 번호": "2022010902508015",
      "색상": "15inch i5-10세대 8G 메모리 + 256G SSD",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스가 부여된 개인 브랜드가 있습니다": "아니요",
      "국경 간 수출을 위한 독점 공급원을 제공할지 여부": "예",
      "에너지 효율 등급": "레벨 1",
      "공급이 특허를 받았나요?": "아니요",
      "유형": "비즈니스 사무실 노트"
    }
  },
  {
    "title": "새로운 15.6 \"하이 엔드 코어 i7 노트북 새로운 디지털 라이트 게임 휴대용 사무실",
    "price": "431,430원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=903138708072&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01lLw0Qf2K0JI8lrnzK_!!2212871969494-0-cib.jpg",
    "detail_specs": {
      "상표": "중립적",
      "모델": "V9",
      "품목 번호": "V9",
      "시장 출시 시간": "2025",
      "공급 카테고리": "스팟상품",
      "브랜드 지역": "국내",
      "CPU 유형": "코어/코어 i7",
      "CPU 주파수": "3.6(GHz）",
      "하드 드라이브 용량": "256G/512G/1TB/2TB",
      "광학 플로피 드라이브 위치": "내장형 광학 드라이브",
      "광학 드라이브 유형": "DVD±RW",
      "화면 크기": "15.6inch",
      "그래픽 카드": "통합 그래픽",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해결책": "풀 HD(1920x1080)",
      "운영 체제": "windows10/11",
      "터치스크린 여부": "일반 터치",
      "두께": "휴대성이 좋고 얇음(17.5mm-21mm)",
      "배터리 수명": "5~7시간",
      "제품 크기": "15.6inch",
      "무게": "1.5-2.0KG",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "1~3일",
      "프록시 가입 지원 여부": "지원하다",
      "판매 후 유형": "3개 보증 저장",
      "보증": "3 년",
      "송장": "고객 서비스에 문의하세요",
      "포장 목록": "컴퓨터/전원 어댑터/사용 설명서",
      "3C 인증서 번호": "2024200902001727",
      "비디오 메모리 용량": "통합 디스플레이 4G",
      "색상": "i7 어드밴스드 그레이 [32g]",
      "메모리 용량": "2TB",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 부여된 개인 상표가 있습니다.": "아니요",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "에너지 효율 등급": "없음",
      "공급원이 특허화되어 있나요?": "아니요",
      "유형": "비즈니스 오피스 노트북, 얇고 가벼운 휴대용 노트북, 게이밍 오디오 및 비디오 노트북"
    }
  },
  {
    "title": "16G 512G 태블릿 PC 패드 프로 Netcom 5G 안드로이드 오피스 엔터테인먼트 게임 학습 기계 2 in 1",
    "price": "38,390원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=802774376104&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01SfSTgZ1pd0aT95eEO_!!2218003985382-0-cib.jpg",
    "detail_specs": {
      "모델": "T50-5G",
      "항목 번호.": "T50-5G",
      "시장에 시간": "2024",
      "소스 카테고리": "재고",
      "운영 체제": "기타 운영 체제",
      "프로세서": "MTK",
      "프로세서 주파수": "다른 주요 주파수",
      "저장 유형": "다른 저장 유형",
      "하드 디스크 용량": "500GB",
      "화면 크기": "10.1 인치",
      "카메라": "3 개의 전면 후면 카메라",
      "블루투스": "지원",
      "통신 기능": "지원 전화",
      "네트워크 유형": "모든 Netcom 4G5G/WiFi",
      "내장 유도": "지능형 중력 유도",
      "프로세서 코어": "여덟 코어",
      "지구력": "9 시간 이상",
      "추가 기능": "기타",
      "언어": "기타",
      "재료/프로세스": "금속 유리",
      "제품 크기": "243mm*162.4mm*7.9m",
      "무게": "0.8k",
      "처리 사용자 정의": "예",
      "인쇄 된 로고": "예",
      "에이전트에 가입 지원 여부": "지원",
      "가장 빠른 배송 시간": "1-3 일",
      "송장": "의사 소통 필요",
      "판매 후 유형": "매장 보증",
      "패킹 리스트": "전체 패키지",
      "3C 인증서 번호": "2022010902489181",
      "색상": "블랙 실버 [재생 속도 분할 화면 버전]]",
      "메모리 용량": "16G 실행 1TB 메모리 [1024G 총] 강하게 추천 [패키지 4]",
      "주요 하류 플랫폼": "기타",
      "주요 판매 지역": "기타",
      "라이센스 개인 상표": "예",
      "국경 간 수출의 원천이 독점적인지 여부": "예",
      "통신 유형": "플러그 가능 카드",
      "통신 장비 액세스 라이센스 번호": "17-G990-223329",
      "라디오 전송 장비 유형 승인 코드": "202215597",
      "유형": "휴대 전화 태블릿"
    }
  },
  {
    "title": "TK 폭발성 어린이 태블릿 IPS 화면 IWAWA 라이브 국경 전자 상거래 선물 어린이 태블릿 7 인치",
    "price": "22,860원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=859954903986&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01WioJ3V1O1uvuafblp_!!938681646-0-cib.jpg",
    "detail_specs": {
      "브랜드": "중립",
      "모델": "Q8",
      "품목 번호": "YJ-Q8",
      "시장 출시 시간": "2024",
      "공급 카테고리": "스팟",
      "운영 체제": "ANDROID",
      "프로세서": "MTK",
      "프로세서 주파수": "1.5GHz",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "8GB",
      "화면 크기": "7inch",
      "카메라": "300k 픽셀",
      "블루투스": "지원하다",
      "통신 기능": "지원되지 않음",
      "네트워크 유형": "WIFI",
      "내장 유도": "지능형 중력 감지",
      "프로세서 코어": "4개 코어",
      "배터리 수명": "5시간 미만",
      "추가 기능": "IPS 화면",
      "언어": "중국어 간체",
      "재료/공예": "플라스틱 껍질",
      "제품 크기": "18CM*12CM*0.7cm",
      "무게": "베어메탈 0.27",
      "처리 및 맞춤화": "예",
      "인쇄된 로고": "할 수 있다",
      "에이전트 가입을 지원합니까?": "지원하다",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장은 제공되지 않습니다",
      "판매 후 유형": "전국 공동보증",
      "포장 목록": "박스, 데이터케이블, 충전기, OTG케이블, 설명서",
      "3C 인증서 번호": "2024011606610568",
      "색깔": "검은색",
      "메모리 용량": "MTK 6735 1/16G 7.0 시스템",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스가 있는 개인 브랜드가 있습니다.": "아니요",
      "국경 간 수출 전용 공급 여부": "예",
      "통신 유형": "카드를 삽입할 수 없습니다",
      "특허 출처 여부": "아니요",
      "무선 송신 장비 모델 승인 코드": "123",
      "유형": "선물 태블릿"
    }
  },
  {
    "title": "서피스 프로 3/4/5 투인원 태블릿 노트북 12.3inch",
    "price": "159,730원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=761519544037&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01yeNV9S2EdncUU5yaD_!!2206675648768-0-cib.jpg",
    "detail_specs": {
      "상표": "Surface",
      "모델": "pro3/4/5",
      "품목 번호": "pro3/4/5",
      "시장 출시 시간": "2018년",
      "공급 카테고리": "스팟상품",
      "운영 체제": "Windows",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "기타 주요 주파수",
      "저장 유형": "솔리드 스테이트 드라이브",
      "하드 드라이브 용량": "128GB",
      "화면 크기": "다른 크기",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원되지",
      "네트워크 유형": "WIFI",
      "내장 센서": "지능형 중력 감지",
      "프로세서 코어": "듀얼 코어",
      "배터리 수명": "5~7시간",
      "추가 기능": "솔리드 스테이트 드라이브",
      "언어": "다른",
      "재료/공정": "금속 공예",
      "제품 크기": "292.1mm*201.42mm*8.45mm",
      "무게": "0.766",
      "맞춤형 처리": "아니요",
      "프록시 가입 지원 여부": "지원되지",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장 제공",
      "판매 후 유형": "매장 보증",
      "포장 목록": "기계 + 충전기",
      "3C 인증서 번호": "2021010902418999",
      "색상": "호스트 + 충전기 + 키보드 + 펜",
      "메모리 용량": "Surface Pro5/i7-7660U/16+512",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스가 부여된 개인 상표가 있습니다.": "아니요",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 불가",
      "공급원이 특허화되어 있나요?": "아니요",
      "유형": "노트북 태블릿"
    }
  },
  {
    "title": "크로스보더 Pad6SPro 태블릿 컴퓨터 10.1인치 3G 통화 2+32 안드로이드 시스템 10",
    "price": "52,560원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=899956063218&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN0177IY6g25mCT4GjeDR_!!2219502187568-0-cib.jpg",
    "detail_specs": {
      "상표": "중립적",
      "모델": "s23",
      "품목 번호": "JDLPB-22",
      "시장 출시 시간": "2024",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "1.66GHz",
      "저장 유형": "솔리드 스테이트 드라이브",
      "하드 드라이브 용량": "16GB",
      "화면 크기": "10.1inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원 전화",
      "네트워크 유형": "4G",
      "내장 센서": "지능형 중력 감지",
      "프로세서 코어": "8개의 코어",
      "배터리 수명": "9시간 이상",
      "추가 기능": "다른",
      "언어": "다른",
      "재료/공정": "금속",
      "제품 크기": "10.1inch 화면",
      "무게": "1KG",
      "맞춤형 처리": "예",
      "로고 인쇄": "할 수 있다",
      "프록시 가입 지원 여부": "지원하다",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장 제공",
      "판매 후 유형": "매장 보증",
      "포장 목록": "전원 데이터 케이블, OTG 케이블, 설명서, 중성색 상자",
      "3C 인증서 번호": "2023161606010136",
      "색상": "파란색",
      "메모리 용량": "호주 규정 [AU] 대외 무역 버전은 국내 사용을 지원하지 않습니다.",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 가능",
      "통신장비망접속허가번호": "02-B432-233237",
      "공급원이 특허화되어 있나요?": "아니요",
      "무선송신평가기기 모델 승인코드": "2021160902135332",
      "유형": "선물 태블릿"
    }
  },
  {
    "title": "Puia70 태블릿 울트라 HD 4K 눈 보호 전체 화면 5g 풀 넷콤 학생 사무실 게임 그리기 온라인 수업",
    "price": "54,390원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=851908583470&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01LgDGjD1Wn6fRzcSjn_!!2215641912832-0-cib.jpg",
    "detail_specs": {
      "브랜드": "지메이파이",
      "품목 번호": "Y3",
      "시장 출시 시간": "2024년",
      "공급 카테고리": "스팟",
      "운영 체제": "ANDROID",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "기타 주요 주파수",
      "저장 유형": "솔리드 스테이트 드라이브",
      "하드 드라이브 용량": "500GB",
      "화면 크기": "다른 크기",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원 전화",
      "네트워크 유형": "전체 넷콤 5G+WiFi",
      "내장 유도": "지능형 중력 감지",
      "프로세서 코어": "8개 코어",
      "배터리 수명": "9시간 이상",
      "추가 기능": "망막 화면",
      "언어": "중국어 번체",
      "재료/공예": "고정밀 CNC",
      "제품 크기": "12inch",
      "무게": "1.25",
      "처리 및 맞춤화": "예",
      "인쇄된 로고": "할 수 있다",
      "에이전트 가입을 지원합니까?": "지원하다",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장 제공",
      "판매 후 유형": "전국 공동보증",
      "포장 목록": "호스트 + 액세서리 + 선물",
      "3C 인증서 번호": "2022201606000228",
      "색깔": "P34 로컬 골드",
      "메모리 용량": "16G 실행 + 1024G 메모리/상위 버전에는 Bluetooth 마우스, 키보드, 가죽 케이스 스타일러스가 함께 제공됩니다.",
      "라이센스가 있는 개인 브랜드가 있습니다.": "예",
      "국경 간 수출 전용 공급 여부": "아니요",
      "통신 유형": "삽입 가능한 카드",
      "통신장비망 접속 허가번호": "17-G682-223168",
      "특허 출처 여부": "아니요",
      "무선 송신 장비 모델 승인 코드": "2024023",
      "유형": "엔터테인먼트 태블릿"
    }
  },
  {
    "title": "국경을 넘는 7inch 태블릿 컴퓨터 안드로이드 블루투스 학습 과외 어린이",
    "price": "34,280원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=816562185986&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01cln3K41VcHmly8gSc_!!2217721932673-0-cib.jpg",
    "detail_specs": {
      "상표": "중립적",
      "모델": "Q88 휴대용 가죽 케이스",
      "품목 번호": "Q88 휴대용 가죽 케이스",
      "시장 출시 시간": "2024",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "1.66GHz",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "16GB",
      "화면 크기": "7inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원되지",
      "네트워크 유형": "WIFI",
      "내장 센서": "거리 센서",
      "프로세서 코어": "8개의 코어",
      "배터리 수명": "5시간 이내",
      "추가 기능": "다른",
      "언어": "다른",
      "재료/공정": "ABS",
      "제품 크기": "7inch",
      "무게": "0.5kg",
      "맞춤형 처리": "예",
      "로고 인쇄": "안됨",
      "프록시 가입 지원 여부": "지원되지",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장은 제공되지 않습니다",
      "판매 후 유형": "매장 보증",
      "포장 목록": "본체 + 충전기 + 데이터 케이블 + 사용 설명서 + 터치펜 + 액정 보호 필름 부착 도구 + OTG 젠더",
      "3C 인증서 번호": "2022010902489181",
      "색상": "하늘색",
      "메모리 용량": "유럽 ​​규정",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 부여된 개인 상표가 있습니다.": "아니요",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 불가",
      "공급원이 특허화되어 있나요?": "아니요",
      "무선송신평가기기 모델 승인코드": "123654",
      "유형": "엔터테인먼트 태블릿"
    }
  },
  {
    "title": "국경을 넘는 새로운 5G 고화질 전체 화면 눈 보호 학생 온라인 수업 학습 기계 특수 태블릿 게임 태블릿 컴퓨터",
    "price": "53,890원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=785556531255&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN013BaBzZ1QPqngry9KC_!!2214707411969-0-cib.jpg",
    "detail_specs": {
      "브랜드": "NIBE",
      "모델": "001",
      "항목 번호.": "001",
      "시장에 시간": "2023",
      "소스 카테고리": "재고",
      "운영 체제": "ANDROID",
      "프로세서": "MTK",
      "프로세서 주파수": "다른 주요 주파수",
      "저장 유형": "플래시 메모리",
      "하드 디스크 용량": "128GB",
      "화면 크기": "10.1 인치",
      "카메라": "듀얼 카메라",
      "블루투스": "지원",
      "통신 기능": "지원 전화",
      "네트워크 유형": "WIFI",
      "내장 유도": "지능형 중력 유도",
      "프로세서 코어": "여덟 코어",
      "지구력": "5-7 시간",
      "추가 기능": "IPS 화면",
      "언어": "기타",
      "재료/프로세스": "금속",
      "제품 크기": "10.1",
      "무게": "450",
      "처리 사용자 정의": "예",
      "인쇄 된 로고": "예",
      "에이전트에 가입 지원 여부": "지원",
      "가장 빠른 배송 시간": "1-3 일",
      "송장": "송장이 제공되지 않음",
      "판매 후 유형": "매장 보증",
      "패킹 리스트": "충전기, 수동, 실버 바늘, 펜",
      "색상": "금",
      "메모리 용량": "국내 맞춤화",
      "주요 하류 플랫폼": "기타",
      "주요 판매 지역": "기타",
      "라이센스 개인 상표": "예",
      "국경 간 수출의 원천이 독점적인지 여부": "예",
      "통신 유형": "플러그 가능 카드",
      "특허 출처 여부": "아니",
      "유형": "휴대 전화 태블릿"
    }
  },
  {
    "title": "2025 새로운 스팟 국경 간 10inch 태블릿 소스 원시 구글 영어 시스템 안드로이드 10 코어 HD 화면",
    "price": "37,710원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=759256059929&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01YDjYSy29wy7Hk7oSE_!!2217171908133-0-cib.jpg",
    "detail_specs": {
      "브랜드": "중립",
      "시장에 시간": "2024",
      "소스 카테고리": "스팟",
      "운영 체제": "ANDROID",
      "프로세서": "MTK",
      "프로세서 주파수": "1.6GHz",
      "저장 유형": "플래시 메모리",
      "하드 디스크 용량": "32GB",
      "화면 크기": "10.1 인치",
      "카메라": "1,300 만 픽셀",
      "블루투스": "지원",
      "통신 기능": "지원 전화",
      "네트워크 유형": "WIFI",
      "내장 유도": "3 축 자이로 스코프",
      "프로세서 코어": "여덟 코어",
      "지구력": "9 시간 이상",
      "추가 기능": "이동식 키보드",
      "언어": "기타",
      "재료/프로세스": "금속",
      "제품 크기": "10.1",
      "무게": "570g",
      "처리 사용자 정의": "예",
      "인쇄 된 로고": "예",
      "에이전트에 가입 지원 여부": "지원",
      "가장 빠른 배송 시간": "1-3 일",
      "송장": "송장 제공",
      "판매 후 유형": "매장 보증",
      "색상": "다크 바이올렛",
      "메모리 용량": "영어 규칙",
      "주요 하류 플랫폼": "LAZADA",
      "주요 판매 지역": "중동",
      "라이센스 개인 상표": "예",
      "국경 간 수출의 원천이 독점적인지 여부": "예",
      "통신 유형": "플러그 가능 카드",
      "라디오 전송 장비 유형 승인 코드": "20230116",
      "유형": "노트북 태블릿"
    }
  },
  {
    "title": "T8PLUS 듀얼 기가비트 포트 3 HDMI2.0 N150 오피스 게임 4K 휴대용 소형 컴퓨터 VS N100",
    "price": "246,800원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=690023206457&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN010IWQqw1ge4rfINAS3_!!3193624166-0-cib.jpg",
    "detail_specs": {
      "상표": "ZXFZ",
      "모델": "T8PLUS",
      "품목 번호": "N150",
      "시장 출시 시간": "2022년 10월 27일",
      "공급 카테고리": "스팟상품",
      "유형": "비즈니스 컴퓨터",
      "운영 체제": "인텔 플랫폼",
      "CPU": "셀러론",
      "CPU 유형": "펜티엄/셀러론",
      "CPU 주파수": "3.4GHz",
      "메모리 용량": "8GB",
      "인터페이스": "HDMI*3  USB*3  LAN*2  DC-TYPE-C*1",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장은 제공되지 않습니다",
      "판매 후 서비스": "매장 내 3가지 보증",
      "보증": "1 년",
      "포장 목록": "어댑터*1 HDMI 케이블*1 호스트*1 사용자 가이드*1",
      "3C 인증서 번호": "2022180901037640",
      "색상": "T8PLUS/N150 16G 1TB"
    }
  },
  {
    "title": "어린이 태블릿 학생 q88 태블릿 선물 태블릿 WIFI 태블릿 크로스 보더",
    "price": "29,250원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=558101775999&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01tVdjSu2JsWY3LPvU3_!!2458409477-0-cib.jpg",
    "detail_specs": {
      "브랜드": "중립",
      "모델": "K17",
      "항목 번호.": "147",
      "시장에 출시 할 시간": "2021",
      "소스 카테고리": "재고",
      "운영 체제": "안드로이드",
      "프로세서": "ARM",
      "프로세서 주파수": "1.2GHz",
      "스토리지 유형": "플래시 메모리",
      "하드 디스크 용량": "4GB",
      "화면 크기": "7 인치",
      "카메라": "듀얼 카메라",
      "블루투스": "지원",
      "통신 기능": "지원되지 않음",
      "네트워크 유형": "WLAN",
      "내장 유도": "지능형 중력 유도",
      "프로세서 코어": "4 코어",
      "지구력": "5-7 시간",
      "추가 기능": "중력 유도",
      "언어": "중국어 간체",
      "재료/공정": "플라스틱",
      "무게": "0.7",
      "처리 사용자 정의": "예",
      "인쇄된 로고": "예",
      "가입하는 에이전트를 지원할지 여부": "지원",
      "가장 빠른 배송 시간": "1-3 일",
      "애프터 타입": "스토어 보증",
      "포장 목록": "박스 충전기 데이터 케이블",
      "3C 인증서 번호": "2020160902268975",
      "색상": "오렌지",
      "메모리 용량": "32g",
      "메인 다운스트림 플랫폼": "기타",
      "주된 판매 지역": "중동",
      "국경 간 내보내기 소스가 배타적인지 여부": "예",
      "통신 유형": "카드를 삽입 할 수 없음",
      "특허 출처 여부": "아니오",
      "유형": "학생 태블릿"
    }
  },
  {
    "title": "새로운 코어 17 세대 노트북 15 \"4G 솔로 게임 오피스 학습 넷북 노트북",
    "price": "141,450원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=794668604669&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01db19yL1toh9PQ6DUS_!!2217768285949-0-cib.jpg",
    "detail_specs": {
      "브랜드": "NEC",
      "공급 카테고리": "스팟",
      "CPU 유형": "코어/코어 i5",
      "하드 드라이브 용량": "512G",
      "화면 크기": "15inch",
      "그래픽 카드": "독립 그래픽 카드",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해상도": "풀 HD(1920x1080)",
      "운영 체제": "Windows 10",
      "터치스크린 여부": "PC 태블릿 투인원",
      "두께": "휴대성과 경량성(17.5mm-21mm)",
      "배터리 수명": "9시간 이상",
      "제품 크기": "화면 13.3/14/15/16 버전 이상",
      "무게": "1.3-1.4kg",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "하루",
      "에이전트 가입을 지원합니까?": "지원하다",
      "판매 후 유형": "전국 공동보증",
      "보증": "3년",
      "송장": "송장 제공",
      "포장 목록": "노트북 호스트 [충전기 및 마우스 포장 포함]",
      "비디오 메모리 용량": "4G",
      "색깔": "검은색",
      "메모리 용량": "1T 솔리드 스테이트 [업그레이드 버전/스크린 터치 스크린]에는 마우스 + 컴퓨터 스탠드 + 컴퓨터 가방이 함께 제공됩니다.",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "중동",
      "라이센스가 있는 개인 브랜드가 있습니다.": "예",
      "국경 간 수출 전용 공급 여부": "예",
      "에너지 효율 등급": "레벨 1",
      "유형": "비즈니스 오피스 노트"
    }
  },
  {
    "title": "12G 독립 디스플레이 i3i5i7 인터넷 카페 e-스포츠 게임 데스크탑 컴퓨터 호스트 디자인 라이브 방송 조립 컴퓨터 전체 배치",
    "price": "178,240원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=665757568213&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN015935M81nfdgxkkiaO_!!2630035117-0-cib.jpg",
    "detail_specs": {
      "브랜드": "기타",
      "모델": "DIY 패키지",
      "항목 번호.": "ZHT12400",
      "시장에 시간": "2022",
      "소스 카테고리": "조립 기계",
      "유형": "게임 컴퓨터",
      "CPU 유형": "인텔/인텔 코어 i7",
      "CPU 주요 주파수": "3.2GHz",
      "메모리 용량": "16GB",
      "하드 디스크 용량": "1TB",
      "디스플레이 유형": "포함되지 않습니다",
      "디스플레이 크기": "포함되지 않습니다",
      "광학 드라이브 유형": "DVD-ROM",
      "그래픽 카드 유형": "이산 그래픽",
      "OEM": "OEM 사용 가능",
      "보증": "1 년",
      "총중량": "10",
      "방열 모드": "공랭식",
      "분류": "DIY 조립 기계",
      "운영 체제": "windows10",
      "가장 빠른 배송 시간": "1-3 일",
      "판매 후 유형": "쇼핑 세 보증",
      "송장": "송장이 제공되지 않음",
      "패킹 리스트": "컴퓨터 호스트",
      "3C 인증서 번호": "2024010901603109",
      "주요 하류 플랫폼": "LAZADA",
      "주요 판매 지역": "중동",
      "라이센스 개인 상표": "예",
      "국경 간 수출의 원천이 독점적인지 여부": "예",
      "에너지 효율 등급": "레벨 1",
      "에너지 효율 기록 번호": "ZH2102P13"
    }
  },
  {
    "title": "2025년 국경을 넘는 글로벌 전문 공급업체",
    "price": "46,850원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=773903930197&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN011YAfSC2MIkMZS5lon_!!2217574169805-0-cib.jpg",
    "detail_specs": {
      "모델": "Pad6 Pro",
      "시장 출시 시간": "2024",
      "운영 체제": "ANDROID",
      "저장 유형": "플래시 메모리",
      "화면 크기": "11.6inch",
      "카메라": "1300만 화소",
      "블루투스": "지원하다",
      "프로세서 코어": "4개의 코어",
      "언어": "영어",
      "처리 사용자 정의": "예",
      "로고 인쇄": "할 수 있다",
      "가장 빠른 배송 시간": "1~3일",
      "색상": "Pad6 플랫 그린 [16 512G]",
      "메모리 용량": "AU 호주 규정 [국내 사용 미지원]]",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스 개인 상표": "아니요",
      "국경 간 수출을 위한 독점 공급원이 있는지 여부": "예",
      "통신 유형": "카드 삽입 가능",
      "특허받은 소스인가요?": "아니요",
      "유형": "선물 태블릿"
    }
  },
  {
    "title": "12inch 학습실 학생 학습용 태블릿 초등학교 및 고등학교 동기화 교과서 고화질 눈 보호 학습용 태블릿",
    "price": "204,880원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=911277785275&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01efYFof29w3TiMppkL_!!2218423938131-0-cib.jpg",
    "detail_specs": {
      "상표": "후조/한지",
      "모델": "C9",
      "품목 번호": "C9 태블릿",
      "시장 출시 시간": "2024",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "기타 주요 주파수",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "6GB",
      "화면 크기": "다른 크기",
      "카메라": "1300만 화소",
      "블루투스": "지원하다",
      "통신 기능": "지원되지",
      "네트워크 유형": "WIFI",
      "내장 센서": "거리 센서",
      "프로세서 코어": "8개 코어",
      "배터리 수명": "7~9시간",
      "추가 기능": "다른",
      "언어": "중국어 간체",
      "재료/공정": "ABS/금속 쉘",
      "제품 크기": "10.1",
      "무게": "450",
      "맞춤형 처리": "예",
      "로고 인쇄": "할 수 있다",
      "프록시 가입 지원 여부": "지원하다",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장 제공",
      "판매 후 유형": "매장 보증",
      "포장 목록": "작업 패키지",
      "3C 인증서 번호": "2023230902002151",
      "색상": "회색",
      "메모리 용량": "학습 시스템 +100위안",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 불가",
      "공급원이 특허화되어 있나요?": "아니요",
      "무선 송신 장비 모델 승인 코드": "20250619",
      "터치스크린 여부": "예",
      "지원되는 언어": "다국어",
      "원산지": "심천, 광둥",
      "유형": "학생용 태블릿"
    }
  },
  {
    "title": "2025년 신형 태블릿 컴퓨터 얇고 가벼운 사무용 4G 카드 삽입 가능 10.1inch 선물용 태블릿 컴퓨터 크로스보더 태블릿",
    "price": "81,130원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=839047813206&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN0151iu6Y1wcHIayv2ZP_!!2215420676328-0-cib.jpg",
    "detail_specs": {
      "상표": "없음",
      "모델": "Y10",
      "품목 번호": "180",
      "시장 출시 시간": "2024년",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "MTK",
      "프로세서 주파수": "1.5GHz",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "32GB",
      "화면 크기": "10.1inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원 전화",
      "네트워크 유형": "4G",
      "내장 센서": "지능형 중력 감지",
      "프로세서 코어": "8개의 코어",
      "배터리 수명": "5~7시간",
      "추가 기능": "IPS 스크린",
      "언어": "다른",
      "재료/공정": "플라스틱",
      "제품 크기": "241.4*160.2*8mm",
      "무게": "0.5kg",
      "맞춤형 처리": "예",
      "로고 인쇄": "할 수 있다",
      "프록시 가입 지원 여부": "지원하다",
      "가장 빠른 배송 시간": "4~7일",
      "송장": "송장 제공",
      "판매 후 유형": "매장 보증",
      "포장 목록": "태블릿, 충전 헤드, 데이터 케이블",
      "색상": "회색(MT6750 CPU)",
      "메모리 용량": "32GB (개인 구매자는 주문하지 마세요. 감사합니다!)",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 가능",
      "통신장비망접속허가번호": "17-C868-241492",
      "특허 유형": "디자인 특허",
      "특허 번호": "2022011606460254",
      "공급원이 특허화되어 있나요?": "예",
      "무선송신평가기기 모델 승인코드": "202218168",
      "유형": "비즈니스 태블릿"
    }
  },
  {
    "title": "정품 코어 i7 노트북 경량 휴대용 대학생 디자인 그리기 게임 도서 비즈니스 사무실",
    "price": "285,360원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=855614770759&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01zXCvda1gmmY16EBTo_!!2217605194185-0-cib.jpg",
    "detail_specs": {
      "브랜드": "치유",
      "모델": "G62",
      "품목 번호": "G62",
      "시장 출시 시간": "2023-10",
      "공급 카테고리": "스팟",
      "브랜드 지역": "국내",
      "CPU 유형": "인텔 셀러론 N5095",
      "CPU 주파수": "2.5Hz",
      "하드 드라이브 용량": "128GB/256GB/512GB/1TB/2TB",
      "광학 플로피 드라이브 위치": "광학 드라이브 없음",
      "광학 드라이브 유형": "없음",
      "화면 크기": "15.6inch",
      "그래픽 카드": "Intel® UHD Graphics",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해상도": "풀 HD(1920x1080)",
      "운영 체제": "WIN10",
      "터치스크린 여부": "비터치",
      "두께": "보통 얇고 가벼운 (21mm-25mm)",
      "배터리 수명": "5시간 미만",
      "제품 크기": "358*228.5*20.1mm",
      "무게": "1.5-2.0KG",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "1~3일",
      "에이전트 가입을 지원합니까?": "지원하다",
      "판매 후 유형": "매장 3개 보증",
      "보증": "1년",
      "포장 목록": "노트북 + 전원 공급 장치 + 상자 + 설명서 + 마우스 + 마우스 패드",
      "3C 인증서 번호": "2024200902001727",
      "비디오 메모리 용량": "공유 시스템 메모리",
      "색깔": "실버 모델 [다목적 슈퍼 버전 - 지문 포함] Core i7 프로세서 + 인텔 다이내믹 가속 그래픽 카드/사무실 학습 + 게임 플레이",
      "메모리 용량": "32GB 메모리 + 1TB 하드 드라이브",
      "라이센스가 있는 개인 브랜드가 있습니다.": "예",
      "에너지 효율 등급": "레벨 1",
      "특허 출처 여부": "아니요",
      "유형": "얇고 휴대성이 뛰어난 노트북"
    }
  },
  {
    "title": "국경 간 태블릿 10.1 인치",
    "price": "28,340원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=865994152182&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01jhmyjY1G5ZB1JlUZI_!!1995990571-0-cib.jpg",
    "detail_specs": {
      "모델": "VP30",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "MTK",
      "프로세서 주파수": "1.6GHz",
      "저장 유형": "플래시 메모리",
      "하드 드라이브 용량": "64GB",
      "화면 크기": "10.1inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원 전화",
      "네트워크 유형": "WIFI",
      "내장 센서": "지능형 중력 감지",
      "프로세서 코어": "8개의 코어",
      "배터리 수명": "5~7시간",
      "추가 기능": "전화 통신",
      "언어": "다른",
      "재료/공정": "알루미늄 합금",
      "제품 크기": "240mm×160mm*7mm",
      "무게": "500g",
      "맞춤형 처리": "예",
      "로고 인쇄": "할 수 있다",
      "프록시 가입 지원 여부": "지원하다",
      "가장 빠른 배송 시간": "4~7일",
      "송장": "송장은 제공되지 않습니다",
      "판매 후 유형": "매장 보증",
      "포장 목록": "베어메탈/수동/충전 케이블/충전 헤드/포장 상자",
      "3C 인증서 번호": "2024200902000911",
      "색상": "[영국 규정] 소량의 경우 고객 서비스에 가격 변경을 요청하세요.",
      "메모리 용량": "2+16G 전용 국경 간 공급(맞춤형, 최소 배치 1,000개, 재고 없음)",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 부여된 개인 상표가 있습니다.": "아니요",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 가능",
      "무선송신평가기기 모델 승인코드": "20230116",
      "유형": "엔터테인먼트 태블릿"
    }
  },
  {
    "title": "코어 14세대 i714700F 컴퓨터 호스트 게임 스튜디오 멀티 오픈 파이 노드 조립 기계 컴퓨터",
    "price": "90,950원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=942430222481&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01Ut3uGI2GwEwphu9Vf_!!947779079-0-cib.jpg",
    "detail_specs": {
      "상표": "지바이항",
      "모델": "호스트 + 디스플레이",
      "품목 번호": "B0022",
      "시장 출시 시간": "2025년 출시",
      "공급 카테고리": "조립 기계",
      "유형": "컴퓨터를 더 열어보세요",
      "CPU 유형": "E5 엑서셀",
      "CPU 주파수": "3.0GHz",
      "메모리 용량": "64GB",
      "하드 드라이브 용량": "1TB",
      "모니터 유형": "액정",
      "모니터 크기": "21inch",
      "광학 드라이브 유형": "포함하지 않음",
      "그래픽 카드 유형": "라이트 머신 카드",
      "OEM": "OEM 불가",
      "보증": "1년",
      "총중량": "12KG",
      "색상": "패키지 10",
      "마더보드 브랜드": "샹성",
      "그래픽 카드 브랜드": "라듐 바람",
      "방열 방식": "4개의 동관 2",
      "분류": "DIY 조립 기계",
      "운영 체제": "윈도우10",
      "가장 빠른 배송 시간": "1~3일",
      "판매 후 유형": "3개 보증 저장",
      "송장": "송장 제공",
      "포장 목록": "주인",
      "3C 인증서 번호": "2024010901615643",
      "국경 간 수출을 위한 독점 공급원인지 여부": "아니요",
      "에너지 효율 수준": "없음"
    }
  },
  {
    "title": "2025 코어 i7 노트북 15.6 셀러론 N5095 얇은 i9 게임 도서 국경 노트북",
    "price": "159,730원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=923710882506&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01olpY7g2FD4DdLMfi4_!!2211714848845-0-cib.jpg",
    "detail_specs": {
      "상표": "다른",
      "모델": "AU51P , NB02 , TU51",
      "품목 번호": "AU51P",
      "시장 출시 시간": "2025",
      "공급 카테고리": "스팟상품",
      "브랜드 지역": "국내",
      "CPU 유형": "셀러론/셀러론",
      "CPU 주파수": "2.8",
      "하드 드라이브 용량": "500GB",
      "광학 플로피 드라이브 위치": "광학 드라이브 없음",
      "화면 크기": "15.6inch",
      "그래픽 카드": "통합 그래픽",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해결책": "풀 HD(1920x1080)",
      "운영 체제": "windows11",
      "터치스크린 여부": "비터치",
      "두께": "휴대성이 좋고 얇음(17.5mm-21mm)",
      "배터리 수명": "5~7시간",
      "제품 크기": "35*23*0.9cm",
      "무게": "1.5-2.0KG",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "1~3일",
      "프록시 가입 지원 여부": "지원하다",
      "판매 후 유형": "3개 보증 저장",
      "보증": "핵심 구성 요소에 대한 1년 보증",
      "송장": "송장 제공 없음",
      "포장 목록": "노트북, 어댑터",
      "색상": "15.6+ 7inch 터치 듀얼 스크린 N100",
      "메모리 용량": "32G+2048G 솔리드 스테이트 드라이브",
      "주요 다운스트림 플랫폼": "독립 방송국",
      "주요 판매지역": "동북아시아",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "에너지 효율 수준": "없음",
      "공급원이 특허화되어 있나요?": "아니요",
      "유형": "울트라북"
    }
  },
  {
    "title": "16 \"N100 코어 i3-N305/1220P 2-in -1 게임 태블릿 노트북 비즈니스 휴대용",
    "price": "482,850원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=754423427446&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01CuzVRZ2AibOATQX4g_!!2201205668237-0-cib.jpg",
    "detail_specs": {
      "브랜드": "BSLAY",
      "모델": "A160",
      "품목 번호": "A160",
      "시장 출시 시간": "2023년",
      "공급 카테고리": "스팟",
      "브랜드 지역": "국내",
      "CPU 유형": "패키지에 따라 다름",
      "CPU 주파수": "1.6",
      "하드 드라이브 용량": "패키지를 따르세요",
      "광학 플로피 드라이브 위치": "광학 드라이브 없음",
      "광학 드라이브 유형": "광학 드라이브 없음",
      "화면 크기": "16inch",
      "그래픽 카드": "통합 그래픽 카드",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해상도": "2560 ×1600",
      "운영 체제": "win10/11",
      "터치스크린 여부": "일반 터치",
      "두께": "9.2mm",
      "배터리 수명": "5~7시간",
      "제품 크기": "길이 359mm* 폭 237mm* 두께 12.4mm",
      "무게": "1.5-2.0KG",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "1~3일",
      "에이전트 가입을 지원합니까?": "지원하다",
      "판매 후 유형": "매장 3개 보증",
      "보증": "1년",
      "송장": "청구서 제공(고객 서비스에 문의)",
      "포장 목록": "노트북, 충전기, 설명서, 포장",
      "3C 인증서 번호": "2024160902093163",
      "비디오 메모리 용량": "공유하다",
      "색깔": "16inch 터치 2-in-1(Core i3-1220P 24G 메모리) 2.5K 화면",
      "메모리 용량": "하드 드라이브 없음(단일 장치는 전송되지 않음)",
      "주요 다운스트림 플랫폼": "LAZADA",
      "주요 판매지역": "중동",
      "라이센스가 있는 개인 브랜드가 있습니다.": "예",
      "국경 간 수출 전용 공급 여부": "예",
      "에너지 효율 등급": "레벨 2",
      "특허 출처 여부": "아니요",
      "유형": "2 in 1 컴퓨터"
    }
  },
  {
    "title": "2025 코어 i7-13620H 독립 그래픽 카드 4G 컴퓨터 비즈니스 사무용 디자인 게임 16inch 게임 노트북",
    "price": "772,370원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=851090460021&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01qxkXmy1EYImLzptFe_!!1677810363-0-cib.jpg",
    "detail_specs": {
      "브랜드": "일곱 다리 노트북",
      "모델": "i7-13620H",
      "품목 번호": "AP55",
      "시장 출시 시간": "2024",
      "공급 카테고리": "스팟",
      "브랜드 지역": "국내",
      "CPU 유형": "코어/코어 i7",
      "CPU 주파수": "4.70GHz",
      "하드 드라이브 용량": "1TB 이상",
      "화면 크기": "16inch",
      "그래픽 카드": "독립 그래픽 카드",
      "무선 네트워크 카드": "무선 네트워크 카드가 있습니다",
      "해상도": "1920*1200",
      "운영 체제": "windows10/11",
      "터치스크린 여부": "비터치",
      "두께": "일반 두께(>25mm)",
      "배터리 수명": "5시간 미만",
      "제품 크기": "356.8x247.2x31.2mm",
      "무게": "2.0-2.5kg",
      "OEM": "OEM 가능",
      "가장 빠른 배송 시간": "1~3일",
      "에이전트 가입을 지원합니까?": "지원하다",
      "판매 후 유형": "매장 3개 보증",
      "보증": "1년",
      "송장": "송장은 제공되지 않습니다",
      "포장 목록": "노트북*1, 설명서*1, 어댑터*1, 색상 상자 포장*1",
      "3C 인증서 번호": "2022010902509184",
      "색깔": "i9-12900HK【MX550 독립 그래픽】",
      "메모리 용량": "32GB+2TB",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 있는 개인 브랜드가 있습니다.": "예",
      "국경 간 수출 전용 공급 여부": "예",
      "에너지 효율 등급": "없음",
      "특허 출처 여부": "아니요",
      "유형": "비즈니스 오피스 노트"
    }
  },
  {
    "title": "Teclast/Teclast P30T 태블릿 풀 핏 스크린 8코어 4G+128GB 안드로이드 태블릿 키보드 가죽 케이스",
    "price": "123,400원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=734584678039&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01RlhRXt2DSWPjkwSMz_!!3497078608-0-cib.jpg",
    "detail_specs": {
      "브랜드": "Teclast/Taipo",
      "모델": "P30T",
      "항목 번호.": "P30T",
      "시장에 시간": "2025년 7월 10일",
      "소스 카테고리": "스팟",
      "운영 체제": "ANDROID",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "1.6GHz",
      "저장 유형": "플래시 메모리",
      "하드 디스크 용량": "128GB",
      "화면 크기": "10.1 인치",
      "카메라": "듀얼 카메라",
      "블루투스": "지원",
      "통신 기능": "지원되지 않음",
      "네트워크 유형": "WIFI",
      "내장 유도": "지능형 중력 유도",
      "프로세서 코어": "여덟 코어",
      "지구력": "5-7 시간",
      "추가 기능": "얼굴 인식",
      "언어": "기타",
      "처리 사용자 정의": "예",
      "인쇄 된 로고": "예",
      "에이전트에 가입 지원 여부": "지원되지 않음",
      "가장 빠른 배송 시간": "1-3 일",
      "송장": "송장이 제공되지 않음",
      "판매 후 유형": "국민 공동 보험",
      "3C 인증서 번호": "2024011606650447",
      "색상": "블랙((가죽 케이스 + 키보드 패키지)",
      "메모리 용량": "4G+128GB",
      "주요 하류 플랫폼": "LAZADA",
      "주요 판매 지역": "북미",
      "라이센스 개인 상표": "아니",
      "국경 간 수출의 원천이 독점적인지 여부": "예",
      "통신 유형": "삽입 할 수없는 카드",
      "특허 출처 여부": "아니",
      "유형": "엔터테인먼트 태블릿"
    }
  },
  {
    "title": "2024 국경 대외 무역 10.1 인치 태블릿 wifi4G 듀얼 카드 풀 넷콤 어린이 학습 태블릿",
    "price": "61,700원",
    "product_link": "https://ssadagu.kr/shop/view.php?platform=1688&num_iid=807655327774&ss_tx=컴퓨터",
    "thumbnail_url": "https://cbu01.alicdn.com/img/ibank/O1CN01dTu19o1WB5kVj3y0K_!!2214057682749-0-cib.jpg",
    "detail_specs": {
      "상표": "다른",
      "모델": "pad6spro",
      "품목 번호": "pad6spro",
      "시장 출시 시간": "2023년",
      "공급 카테고리": "스팟상품",
      "운영 체제": "ANDROID",
      "프로세서": "기타 프로세서",
      "프로세서 주파수": "1.6GHz",
      "저장 유형": "기타 저장 유형",
      "하드 드라이브 용량": "32GB",
      "화면 크기": "10.1inch",
      "카메라": "듀얼 카메라",
      "블루투스": "지원하다",
      "통신 기능": "지원 전화",
      "네트워크 유형": "4G",
      "내장 센서": "지능형 중력 감지",
      "프로세서 코어": "8개의 코어",
      "배터리 수명": "9시간 이상",
      "추가 기능": "전화 통신",
      "언어": "다른",
      "재료/공정": "금속 껍질",
      "제품 크기": "24*16*1",
      "무게": "0.5KG",
      "맞춤형 처리": "예",
      "로고 인쇄": "할 수 있다",
      "프록시 가입 지원 여부": "지원하다",
      "가장 빠른 배송 시간": "1~3일",
      "송장": "송장 제공 없음",
      "판매 후 유형": "매장 보증",
      "포장 목록": "호스트 + 포장 상자 + 충전기 + 데이터 케이블 + OTG 케이블 + 설명서",
      "3C 인증서 번호": "2021011606423774",
      "색상": "녹색",
      "메모리 용량": "영국 규정",
      "주요 다운스트림 플랫폼": "다른",
      "주요 판매지역": "다른",
      "라이센스가 부여된 개인 상표가 있습니다.": "예",
      "국경 간 수출을 위한 독점 공급원인지 여부": "예",
      "통신 유형": "카드 삽입 가능",
      "통신장비망접속허가번호": "17-D321-183525",
      "공급원이 특허화되어 있나요?": "아니요",
      "무선송신평가기기 모델 승인코드": "1",
      "유형": "비즈니스 태블릿"
    }
  }
]

In [6]:
# ssadagu Json 불러오기 , 상품 메타데이터 파싱

# JSON_PATH = "../../crawling_tests/ssadagu_search_results.json"

# with open(JSON_PATH, "r") as f:
#     data = json.load(f)

# products = data["products"]

print(f"총 상품 수: {len(products)}개")

총 상품 수: 30개


In [7]:
# 모든 상품 임베딩 생성 -> emb_matrix & id_map 만들기

emb_list = []
id_map = {}

print("모든 상품 이미지 → MobileNet 임베딩 생성 시작\n")

for idx, item in enumerate(products):
    url = item["thumbnail_url"]

    emb = get_embedding(url)
    if emb is None:
        continue

    emb_list.append(emb)
    id_map[idx] = {
        "index": idx,
        "title": item["title"],
        "price": item["price"],
        "product_link": item["product_link"],
        "thumbnail_url": item["thumbnail_url"]
    }

    print(f"[{idx}] 임베딩 생성 완료")

print("\n 전체 임베딩 생성 완료!")

모든 상품 이미지 → MobileNet 임베딩 생성 시작

[0] 임베딩 생성 완료
[1] 임베딩 생성 완료
[2] 임베딩 생성 완료
[3] 임베딩 생성 완료
[4] 임베딩 생성 완료
[5] 임베딩 생성 완료
[6] 임베딩 생성 완료
[7] 임베딩 생성 완료
[8] 임베딩 생성 완료
[9] 임베딩 생성 완료
[10] 임베딩 생성 완료
[11] 임베딩 생성 완료
[12] 임베딩 생성 완료
[13] 임베딩 생성 완료
[14] 임베딩 생성 완료
[15] 임베딩 생성 완료
[16] 임베딩 생성 완료
[17] 임베딩 생성 완료
[18] 임베딩 생성 완료
[19] 임베딩 생성 완료
[20] 임베딩 생성 완료
[21] 임베딩 생성 완료
[22] 임베딩 생성 완료
[23] 임베딩 생성 완료
[24] 임베딩 생성 완료
[25] 임베딩 생성 완료
이미지 로드 실패: https://cbu01.alicdn.com/img/ibank/O1CN01CuzVRZ2AibOATQX4g_!!2201205668237-0-cib.jpg
사유: 420 Client Error:  for url: https://cbu01.alicdn.com/img/ibank/O1CN01CuzVRZ2AibOATQX4g_!!2201205668237-0-cib.jpg
[27] 임베딩 생성 완료
[28] 임베딩 생성 완료
[29] 임베딩 생성 완료

 전체 임베딩 생성 완료!


In [8]:
# emb_matrix 저장

emb_matrix = np.vstack(emb_list).astype("float32")

# np.save("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/emb_matrix_mobilenet.npy", emb_matrix)

# print("emb_matrix 저장 완료!\n")

In [ ]:
# id_map.json 저장

# with open("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/id_map_mobilenet.json", "w") as f:
#     json.dump(id_map, f, indent=2)

# print("id_map 저장 완료!\n")

💾 id_map 저장 완료!



In [ ]:
import faiss
from langchain_community.vectorstores import FAISS

# FAISS index 저장

dim = EMB_SIZE
index = faiss.IndexFlatL2(dim)

# FAISS 정규화 필수
faiss.normalize_L2(emb_matrix)

index.add(emb_matrix)

faiss.write_index(index, "/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/faiss_index_mobilenet.bin")

print("FAISS index 저장 완료!")

💾 FAISS index 저장 완료!


In [11]:
# 검색 함수

# 로드
emb_matrix = np.load("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/emb_matrix_mobilenet.npy")
index = faiss.read_index("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/faiss_index_mobilenet.bin")

with open("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/id_map_mobilenet.json", "r") as f:
    id_map = json.load(f)

# 정규화
faiss.normalize_L2(emb_matrix)


def search_similar(url, top_k=5):
    print("\n search_similar 실행!")

    t0 = time.time()

    emb = get_embedding(url)
    if emb is None:
        return []

    vec = emb.reshape(1, -1)

    D, I = index.search(vec, top_k)

    results = [id_map[str(i)] for i in I[0]]

    print(f" 실행시간: {round(time.time() - t0, 4)}초")

    return results

In [ ]:
# 테스트 제발 되라.....

test_url = "https://thumbnail.coupangcdn.com/thumbnails/remote/320x320ex/image/retail/images/6671171604579-e3dec662-3729-4133-8fd9-107e14004798.jpg"

results = search_similar(test_url, top_k=5)

print("\n=== 검색 결과 ===")
for r in results:
    print(f"- {r['title']} | ₩{r['price']} | {r['product_link']}")


🔥 search_similar 실행!
